# Relational generation

Generate parent-child tables with valid keys for join tests and BI demos.

In [ ]:
%pip install great-generator

In [ ]:
from great_generator import generate_relational

data = generate_relational(
    tables={
        "customers": {"schema": "customer_id int primary key, customer_name string, email string, region string", "rows": 1000},
        "orders": {"schema": "order_id int primary key, customer_id int references customers.customer_id, order_date date, order_amount double", "rows": 5000},
    },
    relationships=["orders.customer_id -> customers.customer_id"],
    seed=42,
)
customers = data["customers"]
orders = data["orders"]
customers.head(), orders.head()

In [ ]:
joined = orders.merge(customers, on="customer_id", how="left", validate="many_to_one")
assert joined["customer_name"].notna().all()
joined.groupby("region")["order_amount"].sum().sort_values(ascending=False).head()

In [ ]:
customers.to_parquet("output/customers.parquet", index=False)
orders.to_parquet("output/orders.parquet", index=False)